# 06. Simulación de producción por lotes — OralSolutions
Auditoría de solo lectura de SIM_2025_V1: ocho lotes semanales consecutivos (L01–L08), 407 sesiones del hold-out posterior al corte del 1 de agosto de 2025. L07 y L08 contienen deriva artificial de cancelaciones e inasistencias; la fricción se deriva de ambas. No se modifican etiquetas ni snapshots. Se conserva el diseño de 14 lotes, pero se limita la evidencia ejecutada a ocho por costo de ejecución. Los lotes posteriores quedan fuera de este análisis. La operación propuesta es batch; la API sirve cada observación y registra su decisión. No se generan predicciones nuevas en este notebook.

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'datos.py').exists())
sys.path.insert(0, str(ROOT))
from src.datos import conectar_snowflake, consultar_dataframe
from dotenv import load_dotenv
load_dotenv(ROOT/'.env')
SCHEMA = 'ORALSOLUTIONS_DW.MLOPS'
SIM = 'SIM_2025_V1'
FILTRO = f"SIMULACION_ID = '{SIM}' AND LOTE_ID BETWEEN 'L01' AND 'L08'"
DOCS = ROOT/'docs'
DOCS.mkdir(exist_ok=True)
def consulta(sql):
    conn = conectar_snowflake()
    try:
        return consultar_dataframe(conn, sql)
    finally:
        conn.close()
def tabla(nombre):
    return consulta(f'SELECT * FROM {SCHEMA}.{nombre} WHERE {FILTRO} ORDER BY LOTE_ID')


## 1. Control de lotes y maduración
Las etiquetas se evalúan después de 60 días. L01 se evaluó al madurar y L02–L08 en una revisión consolidada el 20 de noviembre de 2025.

In [2]:
lotes = tabla('LOTES_PRODUCCION')
assert len(lotes)==8 and lotes.lote_id.is_unique
assert lotes.estado.eq('evaluado').all()
assert (lotes.total_observaciones == lotes.total_predicciones).all()
assert lotes.total_errores.sum()==0
assert (pd.to_datetime(lotes.fecha_evaluacion_simulada)>=pd.to_datetime(lotes.fecha_maduracion)).all()
display(lotes[['lote_id','fecha_inicio','fecha_fin','fecha_maduracion','fecha_evaluacion_simulada','estado','total_predicciones']])

c:\Users\ANDRES\Desktop\proyecto_final\.venv\Lib\site-packages\snowflake\connector\connection\_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


,lote_id,fecha_inicio,fecha_fin,fecha_maduracion,fecha_evaluacion_simulada,estado,total_predicciones
0,L01,2025-08-01,2025-08-03,2025-10-02,2025-10-02,evaluado,12
1,L02,2025-08-04,2025-08-10,2025-10-09,2025-11-20,evaluado,37
2,L03,2025-08-11,2025-08-17,2025-10-16,2025-11-20,evaluado,53
3,L04,2025-08-18,2025-08-24,2025-10-23,2025-11-20,evaluado,61
4,L05,2025-08-25,2025-08-31,2025-10-30,2025-11-20,evaluado,54
5,L06,2025-09-01,2025-09-07,2025-11-06,2025-11-20,evaluado,68
6,L07,2025-09-08,2025-09-14,2025-11-13,2025-11-20,evaluado,59
7,L08,2025-09-15,2025-09-21,2025-11-20,2025-11-20,evaluado,63


## 2. Predicciones persistidas y trazabilidad

In [3]:
pred = tabla('PREDICCIONES')
assert len(pred)==407 and pred.id_prediccion.is_unique and pred.sesion_key.is_unique
assert pred.origen.eq('simulacion').all() and not pred.uso_respaldo.any()
assert pred.version_modelo.astype(str).eq('1').all()
assert pred.run_id.notna().all() and pred.snapshot_id.notna().all() and pred.hash_dataset.notna().all()
assert pred.probabilidad.between(0,1).all()
assert pred.decision.eq(np.where(pred.probabilidad>=pred.umbral_aplicado,'contactar','no_contactar')).all()
conteo = pred.groupby('lote_id').size()
assert conteo.eq(lotes.set_index('lote_id').total_predicciones).all()
display(pred.groupby(['lote_id','version_modelo','run_id','snapshot_id','hash_dataset'],dropna=False).size().rename('n').reset_index())
display(pred[['id_prediccion','sesion_key','plan_tratamiento_id','fecha_referencia','fecha_maduracion','probabilidad','decision']].head())

c:\Users\ANDRES\Desktop\proyecto_final\.venv\Lib\site-packages\snowflake\connector\connection\_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


,lote_id,version_modelo,run_id,snapshot_id,hash_dataset,n
0,L01,1,c90a1c37aef940278afc025b1d22dc6b,SNAP_SIM_2025_V1,11ae3cc53264511ef486f12d7a3fc0ddf2c6311e3ae7a8...,12
1,L02,1,c90a1c37aef940278afc025b1d22dc6b,SNAP_SIM_2025_V1,11ae3cc53264511ef486f12d7a3fc0ddf2c6311e3ae7a8...,37
2,L03,1,c90a1c37aef940278afc025b1d22dc6b,SNAP_SIM_2025_V1,11ae3cc53264511ef486f12d7a3fc0ddf2c6311e3ae7a8...,53
3,L04,1,c90a1c37aef940278afc025b1d22dc6b,SNAP_SIM_2025_V1,11ae3cc53264511ef486f12d7a3fc0ddf2c6311e3ae7a8...,61
4,L05,1,c90a1c37aef940278afc025b1d22dc6b,SNAP_SIM_2025_V1,11ae3cc53264511ef486f12d7a3fc0ddf2c6311e3ae7a8...,54
5,L06,1,c90a1c37aef940278afc025b1d22dc6b,SNAP_SIM_2025_V1,11ae3cc53264511ef486f12d7a3fc0ddf2c6311e3ae7a8...,68
6,L07,1,c90a1c37aef940278afc025b1d22dc6b,SNAP_SIM_2025_V1,11ae3cc53264511ef486f12d7a3fc0ddf2c6311e3ae7a8...,59
7,L08,1,c90a1c37aef940278afc025b1d22dc6b,SNAP_SIM_2025_V1,11ae3cc53264511ef486f12d7a3fc0ddf2c6311e3ae7a8...,63


,id_prediccion,sesion_key,plan_tratamiento_id,fecha_referencia,fecha_maduracion,probabilidad,decision
0,9ed04794-103b-434f-8ebb-61ea997a356b,220,97,2025-08-01,2025-09-30,0.193516,contactar
1,9d31b7c3-a71a-4442-949b-d7911faddc00,1352,590,2025-08-01,2025-09-30,0.084817,contactar
2,c0825d3f-925e-436c-91b0-4cf00af49bd2,3829,1708,2025-08-01,2025-09-30,0.168514,contactar
3,afda513e-36b2-4362-a2fb-d24153ef690d,4368,1950,2025-08-01,2025-09-30,0.139007,contactar
4,a311326f-d9e3-4458-8485-18719db11012,4563,2036,2025-08-01,2025-09-30,0.194844,contactar


## 3. Telemetría HTTP
L01 fue procesado antes de incorporar la captura HTTP; sus métricas de transporte quedan ausentes, no se imputan como cero.

In [4]:
http = tabla('SOLICITUDES_HTTP')
inmediato = tabla('MONITOREO_INMEDIATO')
display(inmediato[['lote_id','volumen_http','errores_4xx','errores_5xx','latencia_http_p50_ms','latencia_http_p95_ms','latencia_inferencia_p95_ms']])
print('Solicitudes HTTP registradas:',len(http))
print('Versiones registradas:',pred.version_modelo.unique())

c:\Users\ANDRES\Desktop\proyecto_final\.venv\Lib\site-packages\snowflake\connector\connection\_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


,lote_id,volumen_http,errores_4xx,errores_5xx,latencia_http_p50_ms,latencia_http_p95_ms,latencia_inferencia_p95_ms
0,L01,NaN,NaN,NaN,NaN,NaN,71.84780
1,L02,37.0,0.0,0.0,2829.623000,4037.292960,133.63000
2,L03,53.0,0.0,0.0,3002.059150,4546.613179,129.00360
3,L04,61.0,0.0,0.0,2826.233663,3802.229917,55.70200
4,L05,54.0,0.0,0.0,3028.416024,3758.259685,36.87730
5,L06,68.0,0.0,0.0,3010.672279,4096.078447,44.10225
6,L07,59.0,0.0,0.0,2856.360222,4111.295891,33.21830
7,L08,63.0,0.0,0.0,2891.461984,4132.117899,35.56970


Solicitudes HTTP registradas: 395
Versiones registradas: <ArrowStringArray>
['1']
Length: 1, dtype: str


## 4. Exportación de evidencias
Los CSV oficiales incluyen los ocho lotes de la simulación; las demostraciones manuales de rollback y respaldo se conservan separadas.

In [5]:
for nombre, df in [('predicciones',pred),('lotes_produccion',lotes),('solicitudes_http',http)]:
    df.to_csv(DOCS/f'{nombre}.csv',index=False)
demo = consulta(f"SELECT * FROM {SCHEMA}.PREDICCIONES WHERE ORIGEN='demostracion' ORDER BY TIMESTAMP_UTC")
demo.to_csv(DOCS/'predicciones_demostracion.csv',index=False)
print('Evidencias exportadas a docs/')

c:\Users\ANDRES\Desktop\proyecto_final\.venv\Lib\site-packages\snowflake\connector\connection\_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Evidencias exportadas a docs/
